In [3]:
from answer_generation_prompts import VERBOSE

import os
import sys
import json
import re
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import InferenceClient
import torch, gc
from answer_generation_prompts import BASELINE_PROMPT

openai_client = OpenAI() #ensure you define your openai api key
hf_client = InferenceClient() #ensure hf_token w/ finegrained inference permission is defined
cache_lock = threading.Lock()

MAX_WORKERS = 8
user_prompt_template  = """Your task is to judge whether the given response to a question matches a given ground truth answer or not. You are provided with a question, a ground truth response, and the response you need to judge.\n    For a response to "match", it must have at least as much information as the ground-truth. \n    The response can have more information than the ground-truth. It can be more specific (for example, "Labrador" is more specific than "dog"), or have additional possible correct answers. But it must cover everything mentioned in the ground-truth. It is okay if it covers it in different words, i.e. paraphrased. \n    For numeric answers, the relative error, defined as |response - ground truth| / mean(response, ground truth), must be less than 1% for the response to be judged as a correct match. Here, if the ground truth is a specific numeric quantity but the response is a range, then they don\'t match (even if the range contains the ground truth).\n\n    Possible judgments:\n\n    "0": The response does not match the ground-truth answer.\n    "1": The response matches the ground-truth.\n\n    Question: "{question}"\n    Ground truth: "{reference}"\n    Response: "{answer}"\n\n    Your job is to ONLY check whether the given response matches the ground truth answer or not in the context of the question. You DO NOT NEED to assess the correctness of the response. This is part of an automated evaluation process, therefore you MUST OUTPUT your final answer as "0" or "1" in <answer> </answer> tags. End your response with <answer>0</answer> OR <answer>1</answer> TAGS.
"""


judge_prompt = """Your task is to judge whether the given response to a question is correct or not. You are given a question and the response you are judging.
    Possible judgments:
    "0": The response is incorrect. 
    "1": The response is correct. 

    Question: "{question}"
    Response: "{answer}"

    The response should fully answer the question and must not be vague.
    For numeric answers, the relative error, defined as |response - ground truth| / mean(response, ground truth), must be less than 1% for the response to be judged as a correct match. Here, if the ground truth is a specific numeric quantity but the response is a range, then they don't match (even if the range contains the ground truth).

    To the best of your knowledge: Does the provided response answer the question correctly? This is part of an automated evaluation process, therefore you MUST OUTPUT your final answer as "0" or "1" in <answer> </answer> tags.
    
    YOU SHOULD ALWAYS END YOUR RESPONSE WITH <answer>0</answer> OR <answer>1</answer> TAGS.
    """


def answer_question_gpt(record, question_number, cache, cache_file, model="gpt-4.1-mini", PROMPT_TEMPLATE=judge_prompt):

    cache_key = f"{record['question']}"
    with cache_lock:
        if cache_key in cache:
            # Just a small, single-line progress print
            print(f"[{question_number} cached]")
            return cache[cache_key]
    
    response = openai_client.responses.create(
        model=model,
        input=PROMPT_TEMPLATE.format(**record),
        max_output_tokens=2048,
        temperature=0
    )

    text_response = response.output_text.strip()

    with cache_lock:
        cache[cache_key] = text_response
        with open(cache_file, "w") as f:
            json.dump(cache, f, indent=2)
        # print(f"[{question_number} new]")

    return text_response


In [4]:
def extract_answer_tags(response):
        pattern = r'<answer>(.*?)</answer>'
        match = re.search(pattern, response, re.DOTALL | re.IGNORECASE)
        if match:
            content = match.group(1).strip()
            num_match = re.search(r'\b\d+(\.\d+)?\b', content)
            if num_match:
                return num_match.group(0)
            return content
        return response

In [5]:
def generate_answers(question_df, df_type, model, PROMPT_TEMPLATE):
    cache_file = f"gpqa_diamond_cache_{df_type}_{model}_answers.json"

    # Load cache if exists
    if os.path.exists(cache_file):
        with open(cache_file, "r") as f:
            cache = json.load(f)
    else:
        cache = {}
    # Collect questions to process (skip ones already cached)
    to_process = [(i, ex) for i, ex in enumerate(question_df)
                  if f"{ex['question']}" not in cache]

    total = len(to_process)
    progress = [0]  # list so it can be mutated inside threads

    # For storing answers in a dict keyed by index to preserve order
    indexed_results = {}

    def process_item(item):
        idx, example = item
        q_text = example['question']
        ans_text = example['answer']
        ref = example['reference']
        if 'gpt' in model.lower():
            ans = answer_question_gpt(example, idx, cache, cache_file, model, PROMPT_TEMPLATE)
        with cache_lock:
            progress[0] += 1
            # print(f"✓ Answered question {progress[0]}/{total}")
            
            score = extract_answer_tags(ans)
            cache[f"{q_text}"] = [ref, ans, score]
            indexed_results[idx] = (q_text, ref, ans_text, score)
        return ans

    # Process in parallel
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(process_item, item): item for item in to_process}
        for future in tqdm(as_completed(futures), total=len(to_process), desc="Processing"):
            try:
                future.result()
            except Exception as e:
                idx, _ = futures[future]
                print(f"Error processing question {idx}: {e}")

    # Add already cached items to results
    for i, ex in enumerate(question_df):
        key = f"{ex['question']}"
        if key in cache:
            cached_ref, cached_ans, cached_score = cache[key]
            indexed_results[i] = (ex['question'], cached_ref, ex['answer'], cached_score)

    # Save updated cache
    with open(cache_file, "w") as f:
        json.dump(cache, f, indent=2)

    # Sort by original question order
    sorted_results = [indexed_results[i] for i in sorted(indexed_results)]
    answer_df = pd.DataFrame(sorted_results, columns=["question", "reference", "response", "score"])
    print("Processing complete.")
    return answer_df


In [6]:
import argparse
parser = argparse.ArgumentParser()

parser.add_argument('--testee_name', default="Qwen2.5-7B-Instruct", type=str)
parser.add_argument('--matcher_name', default="Qwen3-4B", type=str)
parser.add_argument('--exp', default="baseline", type=str)

#process only so many data points
parser.add_argument('--max_sample_size', default=5, type=int)

parser.add_argument('--testee_prompt', default=BASELINE_PROMPT, type=str)
parser.add_argument('--matcher_prompt', default=judge_prompt, type=str)

parser.add_argument('--temperature_testee', default=0.6, type=float)
parser.add_argument('--max_new_tokens_testee', default=300, type=int)
parser.add_argument('--temperature_matcher', default=0.01, type=float)
parser.add_argument('--max_new_tokens_matcher', default=2048, type=int)

parser.add_argument('--cache_name', default="qwen7b", type=str)
parser.add_argument('--num_retries', default=3, type=int)
parser.add_argument('--retry_tok', default=5000, type=int)

#matcher
parser.add_argument('--model_split', default=False, type=bool)
parser.add_argument('--data_split', default=True, type=bool)


parser.add_argument('--base_qual_data', default="datasets/gpqa_diamond_qualitative.csv", type=str)
parser.add_argument('--base_quant_data', default="datasets/gpqa_diamond_quantitative.csv", type=str)

#paths to read from - matcher
parser.add_argument('--qual_ans', default="/kaggle/input/gpqa-diamond/gpqa_diamond_qual_qwen_answers.csv", type=str)
parser.add_argument('--quant_ans', default="/kaggle/input/gpqa-diamond/gpqa_diamond_quant_qwen_answers.csv", type=str)

#paths to save to
parser.add_argument('--qual_gen_op', default="qwen_qual_baseline_answers.csv", type=str)
parser.add_argument('--quant_gen_op', default="qwen_quant_baseline_answers.csv", type=str)
parser.add_argument('--qual_match_op', default="qwen_qual_baseline_matches.csv", type=str)
parser.add_argument('--quant_match_op', default="qwen_quant_baseline_matches.csv", type=str)


args, unknown = parser.parse_known_args()

args.testee_name = "Qwen2.5-7B-Instruct"
args.matcher_name = "gpt-4.1-mini"
args.testee_prompt = BASELINE_PROMPT
# args.matcher_prompt = judge_prompt
args.cache_name = ["vbqwenmatch", "vbgptmatch", 
                #    "stratqwenmatch", "stratgptmatch", "fwdqwenmatch", "fwdgptmatch"
                   ]
args.model_split = True
args.data_split = True
args.exp = "verbose"
args.max_sample_size = 50000 #whole dataset

args.qual_ans = ["gpqa/verbose/qwen_qual_verbose_answers.csv", 
                "gpqa/verbose/gpt_qual_verbose_answers.csv",  
                #     "gen/qwen_qual_strategic_answers.csv", "gen/gpt_qual_strategic_answers.csv",
                #    "gen/qwen_qual_forward_answers.csv", "gen/gpt_qual_forward_answers.csv"
                   ]

args.quant_ans = ["gpqa/verbose/qwen_quant_verbose_answers.csv", 
                  "gpqa/verbose/gpt_quant_verbose_answers.csv", 
                #     "gen/qwen_quant_strategic_answers.csv", "gen/gpt_quant_strategic_answers.csv",
                #    "gen/qwen_quant_forward_answers.csv", "gen/gpt_quant_forward_answers.csv"
                   ]

# args.qual_gen_op = ["gen/qwen_qual_baseline_answers.csv", "gen/gpt_qual_baseline_answers.csv", 
#                     "gen/qwen_qual_strategic_answers.csv", "gen/gpt_qual_strategic_answers.csv",
#                    "gen/qwen_qual_forward_answers.csv", "gen/gpt_qual_forward_answers.csv"]

# args.quant_gen_op = ["gen/qwen_quant_baseline_answers.csv", "gen/gpt_quant_baseline_answers.csv", 
#                     "gen/qwen_quant_strategic_answers.csv", "gen/gpt_quant_strategic_answers.csv",
#                    "gen/qwen_quant_forward_answers.csv", "gen/gpt_quant_forward_answers.csv"]


#write scores
args.qual_match_op = ["qwen_qual_verbose_scores.csv", "gpt_qual_verbose_scores.csv",
                    #  "scores/match/qwen_qual_strategic_scores.csv", "scores/match/gpt_qual_strategic_scores.csv",
                    #  "scores/match/qwen_qual_forward_scores.csv", "scores/match/gpt_qual_forward_scores.csv"
                     ]

args.quant_match_op = ["qwen_quant_verbose_scores.csv", "gpt_quant_verbose_scores.csv",
                    #  "scores/match/qwen_quant_strategic_scores.csv", "scores/match/gpt_quant_strategic_scores.csv",
                    #  "scores/match/qwen_quant_forward_scores.csv", "scores/match/gpt_quant_forward_scores.csv"
                     ]



# args.qual_gen_op = f"gpqa/{args.exp}/qwen_qual_baseline_answers.csv"
# args.quant_gen_op = f"gpqa/{args.exp}/qwen_qual_baseline_answers.csv"
# args.qual_ans = f"gpqa/{args.exp}/qwen_qual_baseline_answers.csv"
# args.quant_ans = f"gpqa/{args.exp}/qwen_quant_baseline_answers.csv"

# args.qual_match_op = f"scores/gpqa/{args.exp}/{args.matcher_name}/qwen_qual_baseline_matches.csv"
# args.quant_match_op = f"scores/gpqa/{args.exp}/{args.matcher_name}/qwen_quant_baseline_matches.csv"



In [7]:
def get_resp_df(q_df, a_df):
    resp = q_df[['question', 'reference', 'question_mcq']].merge(a_df, on='question')
    resp = resp[['question', 'reference', 'answer']]
    return resp

In [8]:
print(args.matcher_prompt)
qual = pd.read_csv("../datasets/gpqa/gpqa_diamond_qualitative.csv")
quant = pd.read_csv("../datasets/gpqa/gpqa_diamond_quantitative.csv")
for cache_name, qual_ans, quant_ans, qual_match_op, quant_match_op in zip(args.cache_name, 
                                                                          args.qual_ans, args.quant_ans, args.qual_match_op, args.quant_match_op):
    qwen_responses = pd.read_csv(qual_ans)
    qual_responses_qwen = get_resp_df(qual, qwen_responses)
    qwen_responses = pd.read_csv(quant_ans)
    quant_responses_qwen = get_resp_df(quant, qwen_responses)
    
    print(f"Judging {qual_ans}")
    gpt_qual_am_df = generate_answers(qual_responses_qwen.to_dict(orient="records"), cache_name, args.matcher_name, args.matcher_prompt)
        
    gpt_qual_am_df.to_csv(qual_match_op, index=False)
    
    print(f"Judging {quant_ans}")
    gpt_quant_am_df = generate_answers(quant_responses_qwen.to_dict(orient="records"), cache_name, args.matcher_name, args.matcher_prompt)
        
    gpt_quant_am_df.to_csv(quant_match_op, index=False)


Your task is to judge whether the given response to a question is correct or not. You are given a question and the response you are judging.
    Possible judgments:
    "0": The response is incorrect. 
    "1": The response is correct. 

    Question: "{question}"
    Response: "{answer}"

    The response should fully answer the question and must not be vague.
    For numeric answers, the relative error, defined as |response - ground truth| / mean(response, ground truth), must be less than 1% for the response to be judged as a correct match. Here, if the ground truth is a specific numeric quantity but the response is a range, then they don't match (even if the range contains the ground truth).

    To the best of your knowledge: Does the provided response answer the question correctly? This is part of an automated evaluation process, therefore you MUST OUTPUT your final answer as "0" or "1" in <answer> </answer> tags.

    YOU SHOULD ALWAYS END YOUR RESPONSE WITH <answer>0</answer> OR

Processing: 100%|██████████| 106/106 [01:03<00:00,  1.67it/s]


Processing complete.
Judging gpqa/verbose/qwen_quant_verbose_answers.csv


Processing: 100%|██████████| 92/92 [00:49<00:00,  1.86it/s]


Processing complete.
Judging gpqa/verbose/gpt_qual_verbose_answers.csv


Processing: 100%|██████████| 106/106 [00:44<00:00,  2.40it/s]


Processing complete.
Judging gpqa/verbose/gpt_quant_verbose_answers.csv


Processing: 100%|██████████| 92/92 [00:34<00:00,  2.67it/s]

Processing complete.


In [ ]:
# import json
# import pandas as pd


# def changesc(js, cs):
#     with open(js, 'r') as f:
#         scores_data = json.load(f)

#     # Load the CSV file
#     csv_file = cs
#     df = pd.read_csv(csv_file)

#     # Update the score column
#     def get_new_score(row):
#         question = row['question']
#         if question in scores_data:
#             _, ans, _ = scores_data[question]
#             return extract_answer_tags(ans)
#         else:
#             return row['score']  # keep original if not found

#     df['score'] = df.apply(get_new_score, axis=1)

#     # Save the updated CSV
#     df.to_csv(csv_file, index=False)

#     print("Score column updated and saved.")




# for cache_name, qual_ans, quant_ans, qual_match_op, quant_match_op in zip(args.cache_name, 
#                                                                           args.qual_ans, args.quant_ans, args.qual_match_op, args.quant_match_op):
#     jsons = [f"gpqa_diamond_cache_{df_type}_{args.matcher_name}_answers.json" 
#              for df_type in cache_name]
#     csvs = []
    

In [10]:
def mean_accuracy(df):
    valid_scores = df["score"].astype(str) 
    valid = valid_scores[valid_scores.str.len() < 2]
    valid = valid.astype(int)
    accuracy = valid.mean()

    verbose_mask = valid_scores.str.len() > 2
    verbose = df.loc[verbose_mask, "question"] 
    # returns questions that the model didnt reliably score

    return accuracy, valid, verbose
    # return accuracy

In [11]:
paths = args.qual_match_op + args.quant_match_op

In [12]:
paths

['qwen_qual_verbose_scores.csv',
 'gpt_qual_verbose_scores.csv',
 'qwen_quant_verbose_scores.csv',
 'gpt_quant_verbose_scores.csv']

In [18]:
for p in paths:

    df = pd.read_csv("../answer-matching/scores/gpqa_judge/qwen2.5_7b/verbose/" + p)
    
    accuracy, valid, verbose = mean_accuracy(df)
    print(accuracy, len(valid), verbose)



0.5566037735849056 106 Series([], Name: question, dtype: object)
0.3867924528301887 106 Series([], Name: question, dtype: object)
0.21739130434782608 92 Series([], Name: question, dtype: object)
0.15217391304347827 92 Series([], Name: question, dtype: object)


In [ ]:
# 0.33962264150943394 106 Series([], Name: question, dtype: object)
# 0.7264150943396226 106 Series([], Name: question, dtype: object)
# 0.25471698113207547 106 Series([], Name: question, dtype: object)
# 0.4716981132075472 106 Series([], Name: question, dtype: object)
# 0.1509433962264151 106 Series([], Name: question, dtype: object)
# 0.2169811320754717 106 Series([], Name: question, dtype: object)
# 0.1 90 36    You come across an algorithm that gives the fo...
# 41    If an equimolar mixture X of two liquids, whic...
# Name: question, dtype: object
# 0.5108695652173914 92 Series([], Name: question, dtype: object)
# 0.05434782608695652 92 Series([], Name: question, dtype: object)
# 0.15217391304347827 92 Series([], Name: question, dtype: object)
# 0.06521739130434782 92 Series([], Name: question, dtype: object)
# 0.05434782608695652 92 Series([], Name: question, dtype: object)


106

In [ ]:
# record = {
# "question":"""If an equimolar mixture X of two liquids, which decolorizes bromine water, is treated with platinum when heated, then an equimolar mixture Y of two other liquids is formed as a result of disproportionation, which does not decolorize bromine water. Hydrogenation of both mixture X and mixture Y in the presence of platinum under rather severe conditions gives only one substance, a certain hydrocarbon Z (mass fraction of hydrogen is 14.28%), which is a constituent of mixture Y and widely used as a solvent. Substance Z does not react further with hydrogen. There are no conjugated multiple bonds in the molecules of the compounds of mixture X.
# Indicate the total number of hydrogen atoms in two liquids of mixture X.

# """,

# "reference": "18",
# "answer" : """Given that the mass fraction of hydrogen in hydrocarbon Z is 14.28%, and knowing that Z is a constituent of mixture Y, we can deduce that Z has a molecular formula of C5H12 (since the ratio corresponds to the empirical formula of pentane). Since mixture X and Y are equimolar, and there are no conjugated multiple bonds, each liquid in mixture X must also have 12 hydrogen atoms, making the total number of hydrogen atoms in two liquids of mixture X 24."""
# }
# answer_question_gpt(record, 1, {},"test.json", model="gpt-4.1-mini", PROMPT_TEMPLATE=judge_prompt)

"Let's analyze the problem step-by-step:\n\n1. Hydrocarbon Z has a hydrogen mass fraction of 14.28%.  \n   - The mass fraction of hydrogen in a hydrocarbon CxHy is:  \n     \\( \\frac{y \\times 1}{x \\times 12 + y \\times 1} \\)  \n   - Given 14.28% = 0.1428, solve for y/x:  \n     \\( 0.1428 = \\frac{y}{12x + y} \\)  \n     \\( 0.1428(12x + y) = y \\)  \n     \\( 1.7136x + 0.1428y = y \\)  \n     \\( 1.7136x = y - 0.1428y = 0.8572y \\)  \n     \\( y = \\frac{1.7136}{0.8572} x = 2x \\)  \n   So the ratio y/x = 2, meaning the hydrocarbon is an alkane with formula CxH2x+2.  \n   For y = 12, x = 5, so C5H12 (pentane) fits perfectly.\n\n2. Z is a constituent of mixture Y, which is formed by disproportionation of mixture X.\n\n3. Mixture X consists of two liquids, equimolar, both decolorizing bromine water (indicating presence of unsaturation, i.e., double bonds).\n\n4. Mixture Y does not decolorize bromine water, indicating saturated compounds.\n\n5. Hydrogenation of both mixtures X and Y 